# 04 — DeBERTa-v3-base PEFT/LoRA Fine-Tuning with Keras

This Kaggle-ready notebook parameter-efficiently fine-tunes `deberta_v3_base_en` with built-in KerasHub LoRA for balanced five-class Amazon review rating prediction. It mirrors the RoBERTa experiment so validation results are directly comparable, while testing DeBERTa-v3's stronger disentangled-attention encoder.

**Do not use the official test set for model selection.** Labels are mapped from `1..5` to `0..4` for training and converted back for reports.

## Kaggle setup

1. Enable a GPU accelerator in Notebook settings.
2. Add the processed dataset created by `02_preprocessing.ipynb`.
3. Recommended offline-safe setup: choose **Add Input → Models**, search for `keras/deberta_v3`, and attach the Keras model `deberta_v3_base_en` before starting `Save & Run All`. The notebook automatically discovers its local preset under `/kaggle/input`.
4. If no local model is attached, enable Internet so KerasHub can resolve the built-in `deberta_v3_base_en` preset.
5. If automatic dataset discovery finds more than one candidate, set `KAGGLE_DATASET_SLUG` in the configuration cell.

The notebook auto-discovers the attached `train.csv` and `validation.csv`; the recommended dataset is `bert_balanced_50000_per_class` (240,000 train and 10,000 validation rows).

In [ ]:
import os
import subprocess
import sys

os.environ['KERAS_BACKEND'] = 'tensorflow'

try:
    import keras_hub
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'keras-hub'])
    import keras_hub

import keras
import numpy as np
import pandas as pd
import sklearn
import tensorflow as tf

print('TensorFlow:', tf.__version__)
print('Keras:', keras.__version__)
print('KerasHub:', keras_hub.__version__)
print('scikit-learn:', sklearn.__version__)

## Configuration — change paths or experiment settings only here

In [ ]:
from pathlib import Path

# Local attached Kaggle Model is preferred; HF is the online fallback.
MODEL_PRESET = 'deberta_v3_base_en'
LOCAL_MODEL_PRESET = None  # Optional explicit /kaggle/input/... preset directory.
NUM_CLASSES = 5
MAX_LENGTH = 256  # Preserve substantially more review evidence than 128 tokens.
EPOCHS = 4  # EarlyStopping still restores the best validation epoch.
PER_REPLICA_BATCH_SIZE = 8  # Reduce to 4 only if DeBERTa-v3-base exhausts P100 memory.
USE_MULTI_GPU = False  # Keep False for TF 2.20 + KerasHub 0.26 LoRA stability.
LORA_RANK = 16
PEAK_LEARNING_RATE = 4.5e-5  # Matches Microsoft's DeBERTa-v3 small/base fine-tuning range.
END_LEARNING_RATE_RATIO = 0.10
WARMUP_RATIO = 0.10
WEIGHT_DECAY = 0.01
RANDOM_STATE = 42
CHECKPOINT_EVERY_BATCHES = 500

# Set this when auto-discovery is ambiguous, e.g. 'amazon-balanced-bert-data'.
KAGGLE_DATASET_SLUG = None

INPUT_ROOT = Path('/kaggle/input')
OUTPUT_DIR = Path('/kaggle/working/deberta_v3_keras_lora')
CHECKPOINT_DIR = OUTPUT_DIR / 'checkpoints'
BACKUP_DIR = OUTPUT_DIR / 'training_backup'
LOG_DIR = OUTPUT_DIR / 'logs'
for directory in [OUTPUT_DIR, CHECKPOINT_DIR, LOG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

keras.utils.set_random_seed(RANDOM_STATE)
tf.config.experimental.enable_op_determinism()

## GPU, mixed precision, and distributed strategy

In [ ]:
gpus = tf.config.list_physical_devices('GPU')
if not gpus:
    raise RuntimeError('No GPU detected. Enable a Kaggle GPU accelerator before training.')

keras.mixed_precision.set_global_policy('mixed_float16')

# TF 2.20 + KerasHub 0.26 can fail with a bool AddN reduction when
# PipelineModel/LoRA is trained under MirroredStrategy on Kaggle T4x2.
# The default strategy reliably uses GPU:0 without cross-replica reduction.
if USE_MULTI_GPU:
    strategy = tf.distribute.MirroredStrategy()
else:
    strategy = tf.distribute.get_strategy()
GLOBAL_BATCH_SIZE = PER_REPLICA_BATCH_SIZE * strategy.num_replicas_in_sync

print('GPUs:', gpus)
print('Strategy:', type(strategy).__name__)
print('Replicas:', strategy.num_replicas_in_sync)
print('Global batch size:', GLOBAL_BATCH_SIZE)
print('Mixed precision policy:', keras.mixed_precision.global_policy())

## Locate and validate the Kaggle dataset

In [ ]:
def find_dataset_file(filename):
    search_root = INPUT_ROOT / KAGGLE_DATASET_SLUG if KAGGLE_DATASET_SLUG else INPUT_ROOT
    candidates = sorted(search_root.rglob(filename))
    valid_candidates = []
    for path in candidates:
        try:
            columns = pd.read_csv(path, nrows=2).columns
            if {'overall', 'model_input'}.issubset(columns):
                valid_candidates.append(path)
        except Exception:
            continue
    if len(valid_candidates) != 1:
        raise FileNotFoundError(
            f'Expected exactly one valid {filename}; found {valid_candidates}. '
            'Set KAGGLE_DATASET_SLUG explicitly.'
        )
    return valid_candidates[0]

TRAIN_PATH = find_dataset_file('train.csv')
VALIDATION_PATH = find_dataset_file('validation.csv')
print('Train:', TRAIN_PATH)
print('Validation:', VALIDATION_PATH)

In [ ]:
usecols = ['overall', 'model_input']
train_df = pd.read_csv(TRAIN_PATH, usecols=usecols, low_memory=False)
validation_df = pd.read_csv(VALIDATION_PATH, usecols=usecols, low_memory=False)

for name, frame in [('train', train_df), ('validation', validation_df)]:
    assert frame['overall'].between(1, 5).all(), f'Invalid labels in {name}'
    assert frame['model_input'].notna().all(), f'Missing text in {name}'
    assert frame['model_input'].str.strip().ne('').all(), f'Empty text in {name}'
    print(name, frame.shape, frame['overall'].value_counts().sort_index().to_dict())

train_text = train_df['model_input'].astype(str).to_numpy()
validation_text = validation_df['model_input'].astype(str).to_numpy()
train_labels = (train_df['overall'].to_numpy(dtype='int32') - 1)
validation_labels = (validation_df['overall'].to_numpy(dtype='int32') - 1)

## Build efficient `tf.data` pipelines

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
options = tf.data.Options()
options.experimental_deterministic = True

train_ds = (
    tf.data.Dataset.from_tensor_slices((train_text, train_labels))
    .shuffle(len(train_text), seed=RANDOM_STATE, reshuffle_each_iteration=True)
    .batch(GLOBAL_BATCH_SIZE, drop_remainder=True)
    .prefetch(AUTOTUNE)
    .with_options(options)
)
validation_ds = (
    tf.data.Dataset.from_tensor_slices((validation_text, validation_labels))
    .batch(GLOBAL_BATCH_SIZE)
    .prefetch(AUTOTUNE)
    .with_options(options)
)

STEPS_PER_EPOCH = len(train_text) // GLOBAL_BATCH_SIZE
TOTAL_STEPS = STEPS_PER_EPOCH * EPOCHS
WARMUP_STEPS = max(1, int(TOTAL_STEPS * WARMUP_RATIO))
DECAY_STEPS = max(1, TOTAL_STEPS - WARMUP_STEPS)
print({
    'steps_per_epoch': STEPS_PER_EPOCH,
    'total_steps': TOTAL_STEPS,
    'warmup_steps': WARMUP_STEPS,
    'decay_steps': DECAY_STEPS,
})

## DeBERTa-v3-base PEFT/LoRA, AdamW, warmup, and cosine decay

KerasHub LoRA freezes the pretrained DeBERTa-v3 backbone and injects trainable low-rank adapters into attention projections. The classification head remains trainable. Rank 16 supplies useful adaptation capacity while training only a small fraction of the full model.

For single-label multiclass classification, micro F1 equals accuracy. The metric is named `micro_f1` so checkpoint selection matches the official objective. Macro and weighted F1 are calculated after prediction for diagnostics.

In [ ]:
learning_rate = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=0.0,
    warmup_target=PEAK_LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    decay_steps=DECAY_STEPS,
    alpha=END_LEARNING_RATE_RATIO,
)

def resolve_deberta_preset():
    if LOCAL_MODEL_PRESET:
        local_path = Path(LOCAL_MODEL_PRESET)
        if not (local_path / 'config.json').exists():
            raise FileNotFoundError(f'No config.json in LOCAL_MODEL_PRESET={local_path}')
        return str(local_path)

    local_candidates = []
    for config_path in INPUT_ROOT.rglob('config.json'):
        candidate = config_path.parent
        candidate_text = str(candidate).lower()
        if 'deberta' in candidate_text:
            local_candidates.append(candidate)

    local_candidates = sorted(set(local_candidates))
    if len(local_candidates) == 1:
        print('Using attached offline Kaggle Model preset.')
        return str(local_candidates[0])
    if len(local_candidates) > 1:
        raise RuntimeError(
            f'Multiple local DeBERTa presets found: {local_candidates}. '
            'Set LOCAL_MODEL_PRESET explicitly.'
        )

    import socket
    try:
        socket.gethostbyname('huggingface.co')
    except OSError as error:
        raise RuntimeError(
            'No attached DeBERTa-v3 preset was found and Kaggle Internet/DNS is disabled. '
            'Use Add Input -> Models -> keras/deberta_v3 -> deberta_v3_base_en, '
            'then restart and run the notebook again.'
        ) from error
    print('No local preset found; using the online Hugging Face fallback.')
    return MODEL_PRESET

effective_model_preset = resolve_deberta_preset()
print('Loading preset:', effective_model_preset)

with strategy.scope():
    model = keras_hub.models.DebertaV3TextClassifier.from_preset(
        effective_model_preset,
        num_classes=NUM_CLASSES,
        activation=None,
        dropout=0.1,
    )
    model.preprocessor.sequence_length = MAX_LENGTH
    model.backbone.enable_lora(rank=LORA_RANK)

    optimizer = keras.optimizers.AdamW(
        learning_rate=learning_rate,
        weight_decay=WEIGHT_DECAY,
        beta_1=0.9,
        beta_2=0.999,
        epsilon=1e-6,
        global_clipnorm=1.0,
    )
    optimizer.exclude_from_weight_decay(var_names=['bias', 'beta', 'gamma'])
    loss = keras.losses.SparseCategoricalCrossentropy(from_logits=True)
    model.compile(
        optimizer=optimizer,
        loss=loss,
        metrics=[keras.metrics.SparseCategoricalAccuracy(name='micro_f1')],
        jit_compile=False,
    )

trainable_parameters = int(sum(np.prod(variable.shape) for variable in model.trainable_weights))
total_parameters = int(sum(np.prod(variable.shape) for variable in model.weights))
trainable_percentage = 100.0 * trainable_parameters / total_parameters

print(f'Trainable parameters: {trainable_parameters:,}')
print(f'Total parameters: {total_parameters:,}')
print(f'Trainable percentage: {trainable_percentage:.4f}%')
model.summary()

## Fault-tolerant callbacks and checkpoint strategy

`ModelCheckpoint(save_freq=400)` is useful as an emergency weight snapshot, but weights alone do not guarantee exact continuation of optimizer and learning-rate-schedule state. `BackupAndRestore` is therefore the primary resume mechanism and also runs every 400 batches. Even with LoRA, a standard Keras weight checkpoint contains the full model weights; it is intentionally overwritten instead of accumulating many large files.

- `training_backup/`: resumable training state; automatically restored when the same notebook is rerun.
- `latest.weights.h5`: emergency periodic weights.
- `best.weights.h5`: best epoch by validation micro F1.
- `deberta_v3_lora_adapters.lora.h5`: compact final LoRA adapter weights from the backbone.

Kaggle `/kaggle/working` is not durable across a completely discarded session. Save a Kaggle notebook version or download the output directory before ending the session.

In [ ]:
callbacks = [
    keras.callbacks.BackupAndRestore(
        backup_dir=str(BACKUP_DIR),
        save_freq=CHECKPOINT_EVERY_BATCHES,
        double_checkpoint=True,
        delete_checkpoint=False,
    ),
    keras.callbacks.ModelCheckpoint(
        filepath=str(CHECKPOINT_DIR / 'latest.weights.h5'),
        save_weights_only=True,
        save_freq=CHECKPOINT_EVERY_BATCHES,
        verbose=1,
    ),
    keras.callbacks.ModelCheckpoint(
        filepath=str(CHECKPOINT_DIR / 'best.weights.h5'),
        monitor='val_micro_f1',
        mode='max',
        save_best_only=True,
        save_weights_only=True,
        verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor='val_micro_f1',
        mode='max',
        patience=2,
        min_delta=1e-4,
        restore_best_weights=True,
        verbose=1,
    ),
    keras.callbacks.CSVLogger(str(LOG_DIR / 'training.csv'), append=True),
    keras.callbacks.TensorBoard(log_dir=str(LOG_DIR / 'tensorboard'), update_freq=100),
    keras.callbacks.TerminateOnNaN(),
]

## Train

This is the long-running cell. If interrupted, rerun the notebook with the same output files present; `BackupAndRestore` resumes from its latest saved training state.

In [ ]:
history = model.fit(
    train_ds,
    validation_data=validation_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

## Load best weights and compute complete validation metrics

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)

best_weights_path = CHECKPOINT_DIR / 'best.weights.h5'
if best_weights_path.exists():
    model.load_weights(best_weights_path)

validation_logits = model.predict(validation_ds, verbose=1)
validation_prediction_zero_based = np.argmax(validation_logits, axis=1)
validation_prediction = validation_prediction_zero_based + 1
validation_truth = validation_labels + 1

metrics = {
    'accuracy': accuracy_score(validation_truth, validation_prediction),
    'micro_f1': f1_score(validation_truth, validation_prediction, average='micro'),
    'macro_f1': f1_score(validation_truth, validation_prediction, average='macro'),
    'weighted_f1': f1_score(validation_truth, validation_prediction, average='weighted'),
}
print(metrics)
print(classification_report(validation_truth, validation_prediction, digits=4))

In [ ]:
import json
import matplotlib.pyplot as plt
import seaborn as sns

matrix = confusion_matrix(validation_truth, validation_prediction, labels=[1, 2, 3, 4, 5])
plt.figure(figsize=(7, 6))
sns.heatmap(matrix, annot=True, fmt='d', cmap='Blues', xticklabels=[1,2,3,4,5], yticklabels=[1,2,3,4,5])
plt.xlabel('Predicted rating')
plt.ylabel('True rating')
plt.title('DeBERTa-v3 validation confusion matrix')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'confusion_matrix.png', dpi=160)
plt.show()

pd.DataFrame({
    'true_rating': validation_truth,
    'predicted_rating': validation_prediction,
}).to_csv(OUTPUT_DIR / 'validation_predictions.csv', index=False)

(OUTPUT_DIR / 'validation_metrics.json').write_text(
    json.dumps(metrics, indent=2) + '\n', encoding='utf-8'
)

## Save final model artifacts

In [ ]:
# KerasHub requires the adapter filename to end exactly with `.lora.h5`.
model.backbone.save_lora_weights(OUTPUT_DIR / 'deberta_v3_lora_adapters.lora.h5')
model.save(OUTPUT_DIR / 'deberta_v3_rating_classifier.keras')
model.save_to_preset(OUTPUT_DIR / 'deberta_v3_rating_classifier_preset')

parameter_stats = {
    'lora_rank': LORA_RANK,
    'trainable_parameters': trainable_parameters,
    'total_parameters': total_parameters,
    'trainable_percentage': trainable_percentage,
}
(OUTPUT_DIR / 'parameter_stats.json').write_text(
    json.dumps(parameter_stats, indent=2) + '\n', encoding='utf-8'
)

print('Artifacts:')
for path in sorted(OUTPUT_DIR.rglob('*')):
    if path.is_file():
        print(path.relative_to(OUTPUT_DIR), f'{path.stat().st_size / 1024**2:.2f} MB')

## Notes for the next stage

- Select the model by validation micro F1; use macro F1 to diagnose minority/adjacent-rating behaviour.
- Do not tune against the official test set.
- If GPU memory is exhausted, lower `PER_REPLICA_BATCH_SIZE` from 8 to 4; keep `MAX_LENGTH=256` when possible.
- If training is stable but underfits, compare 3 versus 4 epochs in a controlled run.
- Compare this run directly with notebook 03 using the same split, input, seed, sequence length, and metrics.
- LoRA is successful when validation performance remains close to full fine-tuning while the trainable parameter percentage and adapter size are much smaller.